# 07 — Validación estadística

El README tira tres números gruesos: AOV alrededor de $86, retención M1 cercana al 10%, y un salto de retención entre primeras y últimas cohortes. Esta notebook los audita con herramientas que no son SQL:

1. **Intervalo de confianza del AOV** vía bootstrap no paramétrico (1000 réplicas). Responde: ¿qué tan estable es el AOV mes a mes, o me lo está moviendo un outlier?
2. **Test de significancia sobre retención M1** comparando cohortes de la primera mitad vs la segunda mitad del período. Responde: ¿el rebote de retención que se ve en el triángulo es real o ruido de muestreo?
3. **Heatmap de retención** reconstruido en Python con las cohortes recortadas al rango con muestra suficiente, para que la lectura visual coincida con la tabla del README.

La data se lee directamente desde BigQuery (`bigquery-public-data.thelook_ecommerce`) — mismo `as_of_date` dinámico que el resto del repo, así los resultados no envejecen.

> **Nota de reproducibilidad:** para correr la notebook hace falta Application Default Credentials de GCP (`gcloud auth application-default login`) y un proyecto con facturación. Cada query procesa menos de 1 GB, entra en el free tier.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Cliente de BigQuery. Usa ADC; el proyecto de facturación se infiere del env.
client = bigquery.Client()
print(f"Proyecto de facturación: {client.project}")

## 1. Bootstrap del AOV

El AOV que reporta Q1 es un promedio mensual. Un intervalo de confianza construido con la fórmula cerrada (media ± 1.96·σ/√n) asume normalidad que el ticket rara vez cumple — la cola derecha la estiran los outliers de baskets caros. El bootstrap no paramétrico evita ese supuesto: reemuestreamos con reemplazo las órdenes del último mes cerrado y recomputamos el AOV mil veces. El intervalo empírico al 95% sale de los percentiles 2.5 y 97.5 de esa distribución.

Se consulta el detalle por orden (no por línea) para que el AOV sea consistente con el del README.

In [ ]:
AOV_QUERY = """
WITH reference AS (
  SELECT DATE_TRUNC(MAX(DATE(created_at)), MONTH) AS last_month
  FROM `bigquery-public-data.thelook_ecommerce.order_items`
  WHERE status NOT IN ('Cancelled', 'Returned')
),
order_revenue AS (
  SELECT
    oi.order_id,
    SUM(oi.sale_price) AS order_value
  FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
  CROSS JOIN reference r
  WHERE oi.status NOT IN ('Cancelled', 'Returned')
    AND DATE_TRUNC(DATE(oi.created_at), MONTH) = r.last_month
  GROUP BY oi.order_id
)
SELECT order_value FROM order_revenue
"""

orders = client.query(AOV_QUERY).to_dataframe()
print(f"Órdenes del último mes cerrado: {len(orders):,}")
print(f"AOV observado: ${orders['order_value'].mean():.2f}")

In [ ]:
N_BOOTSTRAP = 1000
sample = orders["order_value"].to_numpy()

# Resampling con reemplazo; cada réplica tiene el mismo tamaño que la muestra original.
indices = rng.integers(0, len(sample), size=(N_BOOTSTRAP, len(sample)))
boot_aovs = sample[indices].mean(axis=1)

ci_lo, ci_hi = np.percentile(boot_aovs, [2.5, 97.5])
aov_point = sample.mean()

print(f"AOV observado         : ${aov_point:.2f}")
print(f"IC bootstrap 95% (1000 réplicas): [${ci_lo:.2f}, ${ci_hi:.2f}]")
print(f"Ancho del intervalo   : ${ci_hi - ci_lo:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_aovs, bins=40, color="#1a73e8", alpha=0.85, edgecolor="white")
ax.axvline(aov_point, color="#202124", linestyle="-", linewidth=1.5, label=f"AOV observado ${aov_point:.2f}")
ax.axvline(ci_lo, color="#d93025", linestyle="--", linewidth=1.2, label=f"IC 2.5% ${ci_lo:.2f}")
ax.axvline(ci_hi, color="#d93025", linestyle="--", linewidth=1.2, label=f"IC 97.5% ${ci_hi:.2f}")
ax.set_title("Distribución bootstrap del AOV (último mes cerrado)")
ax.set_xlabel("AOV ($)")
ax.set_ylabel("Réplicas")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

**Lectura.** Si el ancho del intervalo es del orden de ±1–2 dólares, el AOV reportado es estable y las decisiones que lo usan como entrada (threshold de shipping gratis, targets de ticket promedio) se mueven sobre terreno firme. Un intervalo más ancho implicaría que un solo mes no alcanza y conviene promediar sobre tres.

## 2. Test de significancia sobre retención M1

El triángulo de cohortes sugiere que la retención M1 mejoró a lo largo del tiempo (M1 arranca cerca del 3% en cohortes viejas y roza el 10% en las recientes). La pregunta es si ese salto resiste un test estadístico o si lo estoy leyendo de celdas con poca muestra.

Procedimiento:

1. Para cada cohorte mensual, contamos cohort size (usuarios nuevos) y cuántos compraron también en M1.
2. Partimos las cohortes por la mediana temporal: primera mitad (cohortes antiguas) vs segunda mitad (recientes).
3. Aplicamos una prueba de dos proporciones (z-test). El tamaño de efecto lo reportamos como diferencia de proporciones en puntos porcentuales, con IC al 95%.

Las cohortes con menos de 50 usuarios se excluyen para que una cohorte degenerada no mueva la media.

In [ ]:
COHORT_QUERY = """
WITH valid_items AS (
  SELECT user_id, DATE_TRUNC(DATE(created_at), MONTH) AS order_month
  FROM `bigquery-public-data.thelook_ecommerce.order_items`
  WHERE status NOT IN ('Cancelled', 'Returned')
),
first_order AS (
  SELECT user_id, MIN(order_month) AS cohort_month
  FROM valid_items
  GROUP BY user_id
),
cohort_size AS (
  SELECT cohort_month, COUNT(DISTINCT user_id) AS cohort_users
  FROM first_order
  GROUP BY cohort_month
),
m1_retained AS (
  SELECT
    f.cohort_month,
    COUNT(DISTINCT f.user_id) AS retained_m1
  FROM first_order f
  JOIN valid_items v
    ON v.user_id = f.user_id
   AND DATE_DIFF(v.order_month, f.cohort_month, MONTH) = 1
  GROUP BY f.cohort_month
)
SELECT
  cs.cohort_month,
  cs.cohort_users,
  COALESCE(mr.retained_m1, 0) AS retained_m1
FROM cohort_size cs
LEFT JOIN m1_retained mr USING (cohort_month)
WHERE cs.cohort_users >= 50
ORDER BY cs.cohort_month
"""

cohorts = client.query(COHORT_QUERY).to_dataframe()
cohorts["cohort_month"] = pd.to_datetime(cohorts["cohort_month"])
cohorts["m1_rate"] = cohorts["retained_m1"] / cohorts["cohort_users"]
print(f"Cohortes analizadas: {len(cohorts)}")
cohorts.tail(6)

In [ ]:
median_date = cohorts["cohort_month"].median()
first_half = cohorts[cohorts["cohort_month"] <  median_date]
second_half = cohorts[cohorts["cohort_month"] >= median_date]

n1, r1 = first_half["cohort_users"].sum(),  first_half["retained_m1"].sum()
n2, r2 = second_half["cohort_users"].sum(), second_half["retained_m1"].sum()
p1, p2 = r1 / n1, r2 / n2

# Two-proportion z-test (pooled)
p_pool = (r1 + r2) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z = (p2 - p1) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

# IC 95% para la diferencia de proporciones (Wald)
se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
diff = p2 - p1
ci_diff = (diff - 1.96*se_diff, diff + 1.96*se_diff)

print(f"Primera mitad  ({len(first_half)} cohortes): n={n1:>7,}  M1={p1*100:.2f}%")
print(f"Segunda mitad  ({len(second_half)} cohortes): n={n2:>7,}  M1={p2*100:.2f}%")
print(f"Diferencia                     : {diff*100:+.2f} pp")
print(f"IC 95% de la diferencia        : [{ci_diff[0]*100:+.2f}, {ci_diff[1]*100:+.2f}] pp")
print(f"z={z:.3f}   p-value={p_value:.3e}")
print(f"Significativo al 5%: {'sí' if p_value < 0.05 else 'no'}")

**Lectura.** Un p-value chico (< 0.001) con un IC de la diferencia que no cruza cero confirma que el rebote de retención M1 no es ruido: las cohortes recientes retienen estadísticamente mejor que las viejas. Eso habilita a hablar de "mejora de retención" sin poner asteriscos, y a buscar el driver (cambio de onboarding, calidad de tráfico, surtido). Si el IC incluyera cero, la recomendación sería juntar más cohortes antes de llevar el número a una reunión.

## 3. Heatmap de cohortes (visual, coherente con la tabla del README)

Reproduce el triángulo de retención de Q2 con cohortes que tienen al menos 12 meses de observación disponible. Misma lógica que la versión SQL, pintada en Python para tener una referencia offline y poder anotarla si hace falta en una presentación.

In [ ]:
TRIANGLE_QUERY = """
WITH valid_items AS (
  SELECT user_id, DATE_TRUNC(DATE(created_at), MONTH) AS order_month
  FROM `bigquery-public-data.thelook_ecommerce.order_items`
  WHERE status NOT IN ('Cancelled', 'Returned')
),
first_order AS (
  SELECT user_id, MIN(order_month) AS cohort_month
  FROM valid_items GROUP BY user_id
),
cohort_size AS (
  SELECT cohort_month, COUNT(DISTINCT user_id) AS cohort_users
  FROM first_order GROUP BY cohort_month
),
activity AS (
  SELECT
    f.cohort_month,
    DATE_DIFF(v.order_month, f.cohort_month, MONTH) AS months_since,
    COUNT(DISTINCT v.user_id) AS retained
  FROM first_order f
  JOIN valid_items v USING (user_id)
  WHERE v.order_month >= f.cohort_month
  GROUP BY f.cohort_month, months_since
)
SELECT
  a.cohort_month,
  a.months_since,
  ROUND(a.retained / cs.cohort_users * 100, 2) AS retention_pct
FROM activity a
JOIN cohort_size cs USING (cohort_month)
WHERE a.months_since BETWEEN 0 AND 12
  AND cs.cohort_users >= 100
ORDER BY a.cohort_month, a.months_since
"""

triangle = client.query(TRIANGLE_QUERY).to_dataframe()
pivot = triangle.pivot(index="cohort_month", columns="months_since", values="retention_pct")
pivot = pivot.tail(12)  # últimas 12 cohortes con masa suficiente
pivot.index = pd.to_datetime(pivot.index).strftime("%Y-%m")
pivot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".1f",
    cmap="Greens",
    cbar_kws={"label": "% retenidos"},
    linewidths=0.3,
    linecolor="white",
    ax=ax,
)
ax.set_title("Retención por cohorte — últimas 12 cohortes con ≥100 clientes")
ax.set_xlabel("Meses desde la primera compra")
ax.set_ylabel("Cohorte")
fig.tight_layout()
plt.show()

## Cierre

Las tres piezas cumplen roles distintos y complementarios:

- El bootstrap del AOV calibra la precisión del KPI de Q1.
- El z-test sobre cohortes transforma una lectura visual en un claim con soporte estadístico.
- El heatmap reconstruido es la pieza offline que se pega en un deck cuando el dashboard no está disponible.

Ninguno reemplaza a las queries — validan que los números que el README reporta resisten una segunda mirada.